# Brain Connectivity Analysis

Load precision matrices (`iC`) and covariance matrices (`C`) for 4 methods. Nodes are grouped by anatomical position (Posterior / Central / Anterior / Midline) and permuted so groups are contiguous.

In [ ]:
import os

# Must be set before any numerical library is imported
os.environ["OPENBLAS_NUM_THREADS"]   = "1"
os.environ["OMP_NUM_THREADS"]        = "1"
os.environ["MKL_NUM_THREADS"]        = "1"       # If using Intel MKL
os.environ["NUMEXPR_NUM_THREADS"]    = "1"   # If using NumExpr
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"  # macOS Accelerate framework

# For Latex (get path from `which latex`)
latex_path = "/Library/TeX/texbin"
os.environ["PATH"] = latex_path+":" + os.environ["PATH"]

import numpy as np
import scipy as sp
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal,norm
from collections import defaultdict
import warnings
import sklearn.exceptions
from scipy.special import erfinv
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm.notebook import tqdm
from types import SimpleNamespace
import pickle
import plotly.graph_objects as go
import nibabel as nib

from _util import *

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath} \usepackage{amssymb}"
})

warnings.filterwarnings("ignore", category=sklearn.exceptions.ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

np.set_printoptions(edgeitems=30, linewidth=100000, formatter=dict(float=lambda x: "%.3g" % x))
make_plot = lambda n,m,s=[1,1],sharex=False, sharey=False: plt.subplots(n,m,figsize=(m*3*s[0],n*2.5*s[1]),squeeze=False,layout='tight',sharex=sharex, sharey=sharey)

# Data Prep

In [ ]:
# load data
labels    = pd.read_csv("./data/fMRI/CortexSubcortex_ColeAnticevic_NetPartition_wSubcorGSR_parcels_LR_LabelKey_grouped.csv")['Group'].to_numpy()
loc       = pickle.load(open("./data/fMRI/xyz.pkl", "rb"))["coord"]
pconn_img = nib.load('./data/fMRI/CortexSubcortex_ColeAnticevic_NetPartition_wSubcorGSR_CovarianceMatrix.pconn.nii') # <- PATH OF MATRIX, load nib package
S = pconn_img.get_fdata().astype(np.float64)
p = S.shape[0]
print(f"Matrix shape: {S.shape}")
Y = make_samples(cov=S, mean=None, n=int(p))

# Estimation

In [ ]:
import os
import pickle
from functools import partial

iC = {}
C  = {}
nll = {}
runtimes={}

folder = "fMRI"
os.makedirs(f"./results/{folder}", exist_ok=True)

jobs = {
    "aquic-1": partial(compute_aquic, Y, c=1),
    "aquic-5": partial(compute_aquic, Y, c=5),
    "aquic-30": partial(compute_aquic, Y, c=30),
    "cv-glasso": partial(compute_glasso_cv, Y),
    "cv-quic": partial(compute_quic_cv, Y),
    "tiger": partial(compute_tiger, Y),
    "clime": partial(compute_clime, Y),
}

def run_method(name, func, folder, Y):
    path = f"./results/{folder}/{name}.pkl"

    if os.path.exists(path):
        with open(path, "rb") as f:
            res = pickle.load(f)
    else:
        res = func()
        with open(path, "wb") as f:
            pickle.dump(res, f)
    temp = -1.0 * sklearn.utils.extmath.fast_logdet(res.iC) + np.trace(res.iC @ np.cov(Y))  

    return res.iC, res.C, res.runtime, temp

for name, func in jobs.items():
    iC[name], C[name], runtimes[name],nll[name] = run_method(name, func, folder, Y)


methods = list(iC.keys())
print(methods)

In [ ]:
print(labels)

In [ ]:

# group_names = {
# 0: "Visual somatomotor",
# 1: "Attention", # Somatomotor + Auditory
# 2: "Task negative",# CON, DAN, FPN, DMN, Language
# 3: "Cingulo-opercular/salient", # PMM, VMM, ORA
# }

# #cannot reorder this!!!
# label_group_info = np.array([
# ['Primary Visual', 0],
# ['Secondary Visual', 0],
# ['Somatomotor', 0],
# ['Cingulo-Opercular', 3],
# ['Dorsal-attention', 1],
# ['Language', 2],
# ['Frontoparietal', 1],
# ['Auditory', 0],
# ['Default', 2],
# ['Posterior Multimodal', 2],
# ['Ventral Multimodal', 2],
# ['Orbito-Affective', 2],
# ], dtype=object)

# #https://www.biorxiv.org/content/10.1101/2022.09.16.508254v1.full
# group_names = {
#     0: "Default",
#     1: "Visual",
#     2: "Somatomotor",
#     3: "Cingulo-Opercular",
#     4: "Other",
# }

# # #cannot reorder this!!!
# label_group_info = np.array([
#     ['Primary Visual',       1],
#     ['Secondary Visual',     1],
#     ['Somatomotor',          2],
#     ['Cingulo-Opercular',    3],
#     ['Dorsal-attention',     4],
#     ['Language',             4],
#     ['Frontoparietal',       4],
#     ['Auditory',             4],
#     ['Default',              0],
#     ['Posterior Multimodal', 4],
#     ['Ventral Multimodal',   4],
#     ['Orbito-Affective',     4],
# ], dtype=object)



# #https://www.biorxiv.org/content/10.1101/2022.09.16.508254v1.full
# group_names = {
#     0: "A",
#     1: "B",
#     2: "C",
# }

# #cannot reorder this!!!
# label_group_info = np.array([
#     ['Primary Visual', 0],
#     ['Secondary Visual', 0],
#     ['Somatomotor', 1],
#     ['Cingulo-Opercular', 1],
#     ['Dorsal-attention', 0],
#     ['Language', 2],
#     ['Frontoparietal', 2],
#     ['Auditory', 1],
#     ['Default', 2],
#     ['Posterior Multimodal', 2],
#     ['Ventral Multimodal', 2],
#     ['Orbito-Affective', 2],
# ], dtype=object)



group_names = {
    0: "U",
    1: "Center",
}

#cannot reorder this!!!
label_group_info = np.array([
    ['L', 0],
    ['R', 0],
    ['C', 1]
], dtype=object)




# remap every node to its coarse group and sort
label_group = label_group_info[labels, 1].astype(int)
inx         = np.argsort(label_group, kind='stable')

labels      = labels[inx]
label_group = label_group[inx]
loc         = loc[inx]

colors = ['#e41a1c', '#377eb8', '#4daf4a',
          '#984ea3', '#ff7f00', '#ffff33',
          '#a65628', '#f781bf', '#999999',
          '#66c2a5', '#8da0cb', '#a6d854']


# Reorder matrix
for m in methods:
    iC[m] = iC[m][np.ix_(inx, inx)]
    C[m]  = C[m][np.ix_(inx, inx)]

In [ ]:
# 3D plot
fig_gt = go.Figure()
for g in sorted(group_names.keys()):
    mask = label_group == g
    fig_gt.add_trace(go.Scatter3d(
        x=loc[mask, 0], y=loc[mask, 1], z=loc[mask, 2],
        mode='markers',
        marker=dict(size=3, color=colors[g], opacity=0.8),
        name=group_names[g],
    ))

axis_off = dict(showticklabels=False, showgrid=False, zeroline=False, showbackground=False)
fig_gt.update_layout(
    title="True brain parcellation (label_group)",
    height=700, width=900,
    margin=dict(l=0, r=0, t=40, b=0),
    legend_title_text="Group",
    scene=dict(xaxis=axis_off, yaxis=axis_off, zaxis=axis_off),
)
fig_gt.show()

## Matrix Visualization

`log|iC|` (precision) and `log|C|` (covariance) for each method. Red lines mark group boundaries.

In [ ]:
eps  = 1e-12
cmap = "jet"

# global limits for coloring
allv = []
for m in methods:
    allv.append(np.log(np.abs(iC[m]) + eps).ravel())
    allv.append(np.log(np.abs(C[m])  + eps).ravel())
allv = np.concatenate(allv)
vmin, vmax = np.nanmin(allv), np.nanmax(allv)


# plot precision, cov and class partitioning
boundaries = np.where(np.diff(label_group))[0] + 0.5
for m in methods:
    Mi = np.log(np.abs(iC[m]) + eps)
    Mc = np.log(np.abs(C[m])  + eps)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

    im0 = ax[0].imshow(Mi, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="none")
    ax[0].set_title(f"{m}  : $\log |iC|$")
    ax[0].set_xticks([])
    ax[0].set_yticks([])

    im1 = ax[1].imshow(Mc, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="none")
    ax[1].set_title(f"{m} : $\log |C|$")
    ax[1].set_xticks([])
    ax[1].set_yticks([])

    for a in ax:
        for b in boundaries:
            a.axhline(b, color='magenta', linewidth=2)
            a.axvline(b, color='magenta', linewidth=2)

    fig.colorbar(im1, ax=ax)
    plt.show()

In [ ]:
import os                                                   
os.environ['METIS_DLL'] = '/opt/homebrew/lib/libmetis.dylib'
          
import networkx as nx
import metis
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from networkx.algorithms.community.quality import modularity, partition_quality
from scipy.optimize import linear_sum_assignment

k                   = len(group_names.keys())
records             = {}
all_clabels_matched = {}

clabels_set={}
A_set={}

for m in methods:

    D   = np.diag(1.0 / np.sqrt(np.diag(iC[m])))
    iCn =  D @ iC[m] @ D

    # the adjacency matrix A is the  abs( negative values of precision matrix )
    A = np.abs(np.minimum(iCn, 0.0))
    np.fill_diagonal(A, 0.0)

    A_set[m]=A

    G = nx.from_numpy_array(A)
    _, clabels = metis.part_graph(G, nparts=k, ubvec=[1.1],recursive=False, objtype='cut')
    clabels    = np.asarray(clabels, dtype=int)

    clabels_set[m]=clabels

    communities = [set(np.where(clabels == c)[0]) for c in range(k)]
    modularity_val = modularity(G, communities, weight="weight")

    # for each community, find the most common label_group value as the matched group
    comm_to_group = {}
    used = np.zeros(k,dtype=bool) # track already assigned groups, no double assigment                                          
    counts = []
    np.random.seed(seed=0)
    for x in range(k):
        idx                     = np.where(clabels == x)[0] # indices of nodes in community x
        vals                    = label_group[idx] # their true group labels
        count_k                 = np.bincount(vals, minlength=k) + np.random.rand() #add random [0,1] so there are not ties
        counts.append(count_k)

    # counts_mat[i,j] is the number of correct classification if community "i" was mapped to group "j"
    counts_mat = np.array(counts)
    row_ind, col_ind = linear_sum_assignment(counts_mat, maximize=True)
    comm_to_group={}
    for x in range(k): 
        comm_to_group[row_ind[x]] = col_ind[x]

    clabels_matched        = np.array([comm_to_group[c] for c in clabels])
    all_clabels_matched[m] = clabels_matched

    # overall + per-group accuracy
    accuracy = (clabels_matched == label_group).mean()

    records[m] = {
        'nnz/row iC':  round(np.count_nonzero(iCn - np.diag(np.diag(iCn))) / iCn.shape[0], 1),
        'nnz/row A':   round(np.count_nonzero(A) / A.shape[0], 1),
        'modularity':  round(modularity_val, 3),
        'accuracy':    round(accuracy, 3),
        'nll(x1E3)':   round(nll[m]/1000, 3),
        'runtime':     round(runtimes[m], 3),
    }

pd.DataFrame(records).T

In [ ]:

# --- 3D subplots colored by matched group (any number of methods) ---
import math

n_methods = len(methods)
n_cols    = math.ceil(math.sqrt(n_methods))
n_rows    = math.ceil(n_methods / n_cols)

specs = [[{'type': 'scatter3d'} for _ in range(n_cols)] for _ in range(n_rows)]
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    specs=specs,
    subplot_titles=methods + [''] * (n_rows * n_cols - n_methods),
    horizontal_spacing=0.01,
    vertical_spacing=0.02,
)

for i, m in enumerate(methods):
    r, c = i // n_cols + 1, i % n_cols + 1
    cl   = all_clabels_matched[m]
    for g in range(k):
        mask = cl == g
        fig.add_trace(go.Scatter3d(
            x=loc[mask, 0], y=loc[mask, 1], z=loc[mask, 2],
            mode='markers',
            marker=dict(size=3, color=colors[g], opacity=0.8),
            name=group_names[g],
            legendgroup=group_names[g],
            showlegend=(i == 0),
        ), row=r, col=c)

axis_off = dict(showticklabels=False, showgrid=False, zeroline=False, showbackground=False)

fig.update_layout(
    height=300 * n_rows + 60,
    width=300 * n_cols,
    margin=dict(l=0, r=0, t=30, b=60),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.02,
        xanchor='center',
        x=0.5,
        title_text='group',
    ),
)
for idx in range(1, n_rows * n_cols + 1):
    scene = f"scene{'' if idx == 1 else idx}"
    fig.update_layout(**{scene: dict(xaxis=axis_off, yaxis=axis_off, zaxis=axis_off)})

fig.show()

In [ ]:
plt.plot(clabels_set['aquic-30'])

In [ ]:
print(A.shape)